In [1]:
import requests
import time
import json
from tqdm import *
import pandas as pd
import os
import sys
sys.path.append("../../source/")
from concurrent.futures import ThreadPoolExecutor, as_completed
from llm_agent import LLMClient
from IntentAgent import IntentAgent
from RewriteQueryAgent import RewriteQueryAgent
from RagEvidenceExtractAgent import RagEvidenceExtractAgent
from CompleteRagAgent import CompleteRagAgent
from BriefAagent import BriefAagent
from PlannerAgent import PlannerAgent
from DraftWriterAgent import DraftWriterAgent
from DraftCriticleAgent import DraftCriticleAgent
from CompleteWriterAgent import CompleteWriterAgent
from WebRetriever import web_retriever
from WebSummaryAgent import WebSummaryAgent
from PlannerCriticleAgent import PlannerCriticleAgent
from CompletePlannerAgent import CompletePlannerAgent
from creat_class_from_config import create_class_from_config


import logging

# 基础配置
logging.basicConfig(
    level=logging.DEBUG,  # 打印级别：DEBUG < INFO < WARNING < ERROR < CRITICAL
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
# 获取日志对象
logger = logging.getLogger("deepagent")

# from dag import DAG, DAGNode
def read_conf(path):
    with open(path) as f:
        agent_prompt = json.load(f)
        return agent_prompt
    
def edit_distance(s1: str, s2: str) -> int:
    """空间优化版编辑距离"""
    if len(s1) < len(s2):
        s1, s2 = s2, s1
    
    m, n = len(s1), len(s2)
    prev = list(range(n + 1))
    
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                cost = 0
            else:
                cost = 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    
    return prev[n]

def longest_common_subsequence(s1: str, s2: str) -> int:
    """最长公共子序列长度"""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    
    return dp[m][n]

def same_ratio(s1: str, s2: str) -> dict:
    """计算两种相同成分占比"""
    if s1 == s2:
        return {"lcs_ratio": 1.0, "edit_ratio": 1.0}
    if not s1 or not s2:
        return {"lcs_ratio": 0.0, "edit_ratio": 0.0}
    
    lcs_len = longest_common_subsequence(s1, s2)
    edit_dist = edit_distance(s1, s2)
    max_len = max(len(s1), len(s2))
    
    return {
        "lcs_ratio": lcs_len / max_len,
        "edit_ratio": (max_len - edit_dist) / max_len,
        "lcs_length": lcs_len,
        "edit_distance": edit_dist,
        "max_length": max_len
    }

/home/work/anaconda3/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.8' currently installed).
  warnings.warn(msg, UserWarning)


In [2]:
def intent_router(memory):
    if "经典句子" in memory.get_intent():
        return "RAG"
    elif "工作写作" in memory.get_intent():
        return "DeepAgent"
    else:
        return 'create'

In [3]:
def rag_retrival_result(memory):
    rewrite_llm = RewriteQueryAgent(prompt=prompts["rewrite"], doc="rewrite")
    rewrite_llm.exceute(llm_client, memory)
    if memory.get_need_search() == "0":
        print("need_search", memory.get_need_search())
        return memory
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "query改写  " + "".join(memory.get_raise_error()))
        return memory
    all_result = {}
    tmp_value = []
    for query in memory.get_search_query():
        result = web_retriever(query)
        for k, v in result.items():
            need = True
            for tv in tmp_value:
                srtv = same_ratio(v, tv)
                if srtv["edit_ratio"] > 0.66:
                    need = False
                    break
            tmp_value.append(v)
            if need:
                all_result[query +"_" + k] = v
    memory.set_evdences(json.dumps(all_result, ensure_ascii=False))
    ragextract_llm = RagEvidenceExtractAgent(prompt=prompts["rag_extract"], doc="extract")
    ragextract_llm.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "rag extract " + "".join(memory.get_raise_error()))
        return memory
    comrag_llm = CompleteRagAgent(prompt=prompts["rag_complete"], doc="complete")
    comrag_llm = comrag_llm.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "rag complete " + "".join(memory.get_raise_error()))
        return memory

    

In [ ]:
def deep_agent_rag(memory, sence):
    brief = BriefAagent(prompt=prompts["brief"], doc="brief")
    brief.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "brief " + "".join(memory.get_raise_error()))
        return memory
    memory.set_main_goal(memory.get_brief()["main_goal"])
    memory.set_success_criteria(str(memory.get_brief()["success_criteria"]))
    planner = PlannerParser(prompt=prompts["planner"], doc="planner")
    planner.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "planner " + "".join(memory.get_raise_error()))
        return memory
    draft = DraftWriterAgent(prompt=prompts["draft"], doc="draft")
    draft.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "draft " + "".join(memory.get_raise_error()))
        return memory

In [ ]:
def deep_agent_without_rag(memory, sence):
    brief = BriefAagent(prompt=prompts["brief"], doc="brief")
    brief.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "brief " + "".join(memory.get_raise_error()))
        return memory
    memory.set_main_goal(memory.get_brief()["main_goal"])
    memory.set_success_criteria(str(memory.get_brief()["success_criteria"]))
    planner = PlannerAgent(prompt=prompts["planner"], doc="planner")
    planner.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "planner " + "".join(memory.get_raise_error()))
        return memory

    draft = DraftWriterAgent(prompt=prompts["draft"], doc="draft")
    draft.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "draft " + "".join(memory.get_raise_error()))
        return memory
    ### 下面两个 criticle 
    print("以下脚本执行容易出问题-" * 66)
    return 
    draft_criticle = DraftCriticleAgent(prompt=prompts["draft_criticle"], doc="draft_criticle")
    draft_criticle.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "draft critile" + "".join(memory.get_raise_error()))
        return memory
    complete_write = CompleteWriterAgent(prompt=prompts["complete_writer"], doc="complete_writer")
    complete_write.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "complete write" + "".join(memory.get_raise_error()))
        return memory

In [ ]:
def sub_generate_result(memory, sence):
    if sence == "":
        
    

In [4]:
def generate_result(outpath, query):
    prompts = read_conf("agent_prompt.json")
    llm_client = LLMClient()
    MemerySystem = create_class_from_config("agent_class.json")
    memory = MemerySystem()
    memory.set_query(query)
#     intent_llm = IntentAgent(prompt=prompts["intent"], doc="intent")
#     intent_llm.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "意图识别  " + "".join(memory.get_raise_error()))
    sence = intent_router(memory)
    print("sence", sence)
    
    ###  rag
    rag_retrival_result()
    
    ## deep_agent_rag



    ###  deep_agent_without_rag
    

    return memory

def main(inpath, outpath):
    remaining_queries = []
    dataframe = pd.read_excel(inpath)
    for indx, row in dataframe.iterrows():
        query = row['Query']
        remaining_queries.append(query)
    print(remaining_queries[1])
    print(remaining_queries[2])
    with open(outpath, 'a') as f:
        with ThreadPoolExecutor(max_workers=2) as executor:
            # 提交任务给线程池
            futures = {executor.submit(generate_result,outpath, query_prompt): query_prompt for query_prompt in remaining_queries[132:]}
            for future in tqdm(as_completed(futures), total=len(remaining_queries)):
                try:
                    memory = future.result()
                    # 立即将结果写入文件
                    f.write(json.dumps(memory.to_dict(), ensure_ascii=False) + '\n')
                    # f.flush()
                except Exception as e:
                    print(f" 产生了一个异常: {e}")

In [ ]:
if __name__ == "__main__":
    inpath = "DeepAgent/data/RAG/query_RAG检索.xlsx"
    outpath = "DeepAgent/data/RAG/query_RAG检索_result.json"
    main(inpath, outpath)

2026-06-09 23:30:44 - urllib3.connectionpool - DEBUG - http://10.11.175.3:80 "POST /tianchi/chat/completions HTTP/1.1" 200 631


query:  人生感悟的经典句子
RagExtractAgent [{'idx': '人生哲理名言警句_chunk_2', 'confidence': 'high', 'result': '人生本就是一幕戏，每个人都是主角，德是一种短暂，诗是一种境界，使人成熟的不是岁月而是经历。'}]
人生感悟的经典句子
阿嬷最经典十句话
sence create


  0%|          | 0/20000 [00:00<?, ?it/s]2026-06-09 23:30:47 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): 10.11.175.3:80
2026-06-09 23:30:49 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): 10.11.175.3:80
2026-06-09 23:30:52 - urllib3.connectionpool - DEBUG - http://10.11.175.3:80 "POST /tianchi/chat/completions HTTP/1.1" 200 760


query:  人生感悟的经典句子
CompleteRagAgent {'thought': "用户query为'生命无价，一定要珍爱'，这是一个关于生命珍贵性的陈述。我需要从Evidence中找到直接包含或最接近此含义的句子。经过检索，在'生命体会金句_chunk_3'中找到了完全一致的原文句子。因此，直接抽取该句作为结果。", 'result': ['生命无价，一定要珍爱。']}
sence create


2026-06-09 23:30:55 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): 10.11.175.3:80
2026-06-09 23:30:58 - urllib3.connectionpool - DEBUG - http://10.11.175.3:80 "POST /tianchi/chat/completions HTTP/1.1" 200 None


...

query:  快乐老八三个儿媳妇的经典语录
RagExtractAgent [{'idx': '快乐老八 三个儿媳妇 经典语录_chunk_1', 'confidence': 'high', 'result': '家中三儿媳，真是我们的福气，快乐与幸福，一直围绕身旁；愿你永远快乐，我们大家庭的幸福源泉；家有仨儿媳，真是好福气啊，每天都快快乐乐的，真幸福；家里热热闹闹的，每天心情都好好；仨儿媳，福气多多，快乐常伴，幸福围绕；家有你们，欢声笑语不断，幸福指数飙升；三个儿媳都是宝，快乐幸福身边绕；幸福日子乐悠悠，儿媳相伴好彩头；家里有三个儿媳，每天都充满了欢声笑语，真是太好了，幸福感满满当当；儿媳们孝顺懂事，又能干，我们真是有福气啊；家里有三个儿媳，真是热闹，每天欢声笑语，好幸福；看着你们快乐，就是幸福的源泉；三个儿媳好快乐，幸福美满乐无边，日子红火喜洋洋；家中有你们真幸福，欢声笑语乐淘淘；你是快乐制造机，有你在家欢乐多，幸福美满乐无边；我们心里乐开花，幸福日子顶呱呱。'}, {'idx': '快乐老八 三个儿媳妇 经典语录_chunk_2', 'confidence': 'high', 'result': '三个儿媳都好棒，带给我无尽的欢笑与温暖；每次看到她们的笑脸，就感觉世界都亮了；我家三个儿媳，真是人见人爱花见花开呀；每天看着她们，生活里都是孩子们的欢声笑语；我这三个儿媳呀，宝贝们是我的骄傲和幸福源泉；有她们在，家里每天都充满了欢声笑语和温馨；我的三个儿媳呀，带给我满满的幸福和快乐；看着她们的笑容，我感受到了生活的美好和温馨；儿媳们好贴心，带给我无尽的欢乐时光；看着她们长大，幸福就藏在生活的点滴里；三个儿媳，你们是我生活中的小太阳，带给我无尽的温暖和快乐；看着你们每天开开心心，我也感觉生活充满了希望和动力；有你们，我是幸福的，每天欢声笑语不断，感恩有你们；你们三个是我最大的财富，有你们陪伴的每一天都是幸福的；三个儿媳，你们是我快乐的源泉，每一天都充满了欢声笑语；有你们真好，我的生活因你们而更加幸福完整，感恩遇见。'}, {'idx': '快乐老八 儿媳妇 名言_chunk_1', 'confidence': 'low', 'result': '因版权方要求该歌曲需付费，购买即可无限畅享~ 立即购买'}, {'idx': '快乐老八 儿媳妇 名言_chunk_2',

2026-06-09 23:46:07 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): 10.11.175.3:80
2026-06-09 23:46:19 - urllib3.connectionpool - DEBUG - http://10.11.175.3:80 "POST /tianchi/chat/completions HTTP/1.1" 200 None


query:  快乐老八三个儿媳妇的经典语录
CompleteRagAgent {'thought': "用户query是'快乐老八 三个儿媳妇 经典语录'，需要从Evidence中抽取与'快乐老八'、'三个儿媳妇'以及'经典语录'相关的直接描述性语句。Evidence中chunk_1和chunk_2包含了大量关于三个儿媳带来快乐和幸福的表述，符合'经典语录'的风格。chunk_3和chunk_4虽然标题包含'快乐老八 儿媳妇 名言'，但内容主要是老八与其他人的对话片段，并未直接描述三个儿媳的快乐场景，与query核心'三个儿媳妇'的关联度较低。因此，优先从chunk_1和chunk_2中抽取直接表达三个儿媳带来快乐、幸福、欢声笑语的句子。", 'result': ['家里有三个儿媳,每天都充满了欢声笑语,真是太好了,幸福感满满当当的', '家里有三个儿媳,真是热闹,每天欢声笑语,好幸福', '三个儿媳都是宝,快乐幸福身边绕', '家有仨儿媳,真是好福气啊,每天都快快乐乐的,真幸福', '有她们在,家里每天都充满了欢声笑语和温馨', '我的三个儿媳呀,带给我满满的幸福和快乐', '三个儿媳,你们是我生活中的小太阳,带给我无尽的温暖和快乐', '三个儿媳,你们是我快乐的源泉,每一天都充满了欢声笑语']}
sence create


2026-06-09 23:46:20 - urllib3.connectionpool - DEBUG - http://10.11.175.3:80 "POST /tianchi/chat/completions HTTP/1.1" 200 None


query:  阿嬷的情书最经典十句话
RagExtractAgent [{'idx': 'chunk_1', 'confidence': 'medium', 'result': '给阿妈的情书，里面阿妈珍藏数十年的侨批家书，并非出自阿公之手，字句间的情谊却胜过了世间万千的情书。妻书柔崭新安康，随现记200元，我一切无恙，生意昌盛。行船入夜，江上升明月，月圆如玉坠，仿若身在故乡，似与你并肩共赏，江海万里，心中念腻，便不觉遥远。湄南河畔木棉花盛开，像极了家乡的春天，压了一朵的信中，望你也能闻到花香。近来我学会了你的名，学会了你的名，虽然辽草，努力数日，定会成功。只短情长，浮为珍，重夫暮生。'}, {'idx': 'chunk_2', 'confidence': 'medium', 'result': '电影《给阿嬷的情书》十句经典台词，展现潮汕文化的独特魅力。最近很流行的电影《给阿嬷的情书》。看哭全网。一封跨越半世纪的书信，藏着最深沉的爱与守候。其中的经典台词，不仅体现了对亲情情义的深刻探讨，也展现了潮汕文化的独特魅力。分享给你。信在，人就在薄薄纸页载不动山海，思念却让分离的人从未真正走远。江海万里，心中念你便不觉遥远。表达跨越山海的牵挂，只要心中惦念，距离不再是阻碍。做人得有情义，无情无义的人不能交往。你在那边好着呢，我也好着呢。善意的谎言，让相思有了双向奔赴的形状。平安当大赚，家人无病无灾，岁岁平安才是最大福气。我等的不是钱，是一句我回来了。行船入夜，恰江上升明月，似与你并肩共赏。谁言女子之肩膀不够伟岸，为母则刚，恰似你的样子。赞颂阿嬷在风雨岁月里，独自撑家养育儿女的坚韧，彰显女性独有的温柔与力量。下南洋一程，隔世间半生。道尽先辈背井离乡的无奈，一场谋生的远行，酿成半生别离，浓缩了一代人的乡愁。纸短情长，伏惟珍重。简洁而深情，表达书信虽短，但情意绵长，愿对方珍重。'}, {'idx': 'chunk_4', 'confidence': 'high', 'result': '2026最火潮汕电影《给阿嬷的情书》中经典语录(完整版) 1. 吾妻淑柔,展信安康。随信寄两百银,我一切无恙,生意昌顺。行船入夜,恰江上升明月,圆如玉坠,仿若身在故乡,似与你并肩共赏。江海万里,心中念你,便不觉得遥远。——木生 2. 阿嬷说:做人得有情有义,无情无义的人不能交往。 3. 一纸侨批

### RAG